In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from statsmodels.formula.api import ols
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
import seaborn as sns

In [ ]:
df = pd.read_csv("/Users/mahinbindra/Downloads/Group Project Data-SFD Data-Toronto Police copy.csv")
df.info()

In [ ]:
df.drop(columns=["X", "Y", "OCC_DATE", "OCC_DOY"], inplace=True)

In [ ]:
df

In [ ]:
df.set_index('OBJECTID', inplace=True)

In [ ]:
df

In [ ]:
df2 = df[df!='NSA']
df2.dropna(inplace=True)
df2.isna().any().any()

In [ ]:
df2.head(20)

In [ ]:
neighbour_group = df2.groupby(["NEIGHBOURHOOD_158"])[["EVENT_UNIQUE_ID"]].count().sort_values(by = ["EVENT_UNIQUE_ID"], ascending=False).reset_index()
neighbour_group.rename(columns={'EVENT_UNIQUE_ID': 'TOTAL CRIMES'}, inplace=True)
neighbour_group

In [ ]:
df3 = pd.DataFrame(neighbour_group)

In [ ]:
df3

In [ ]:
df3['High_crime'] = (df3['TOTAL CRIMES'] > 46).astype(int)
df3['Low_crime'] = (df3['TOTAL CRIMES'] < 9).astype(int)
df3

In [ ]:
df3.describe(include='all')

In [ ]:
df3['TOTAL CRIMES'].unique()

In [ ]:
def total_crimes_cat(x):
    if x >= 0 and x <= 9:
        return 0
    elif x > 9 and x <= 21:
        return 1
    elif x >= 21 and x <= 45:
        return 2
    elif x >= 46:
        return 3

In [ ]:
df3['total_crime_cat'] = df3['TOTAL CRIMES'].apply(total_crimes_cat)

In [ ]:
df3.head(50)

In [ ]:
sns.countplot(x='total_crime_cat', data=df3)
plt.show()

# More detailed exploration
sns.pairplot(df3, hue='total_crime_cat')
plt.show()

In [ ]:
df2.info()

In [ ]:
df2['NEIGHBOURHOOD_158'].unique()

In [ ]:
df2.head(1)

In [ ]:
sns.lmplot(x='temp', y='humidity', hue = 'casual_cat', data=df)

In [ ]:
df2.info()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier #import classifier, not regressor
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_curve, auc, precision_recall_curve
from sklearn.pipeline import Pipeline


In [ ]:
cat_columns = ['OCC_MONTH','OCC_DOW','OCC_TIME_RANGE', 'DIVISION']
num_columns = ['OCC_YEAR','OCC_DAY','OCC_HOUR','DEATH', 'INJURIES']
target = 'LAT_WGS84'

In [ ]:
cat_transformer = OneHotEncoder(handle_unknown='ignore')
num_transformer = StandardScaler()
preprocessor = ColumnTransformer(transformers = [('cat',cat_transformer, cat_columns),
                                                 ('num', num_transformer, num_columns)])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df2[cat_columns + num_columns], df2[target], test_size=0.2, random_state=42)

In [ ]:
x_train_transformed = preprocessor.fit_transform(X_train)
x_test_transformed = preprocessor.fit_transform(X_test)

In [ ]:
x_train_transformed.shape

In [ ]:
x_test_transformed.shape

In [ ]:
# Assuming you have already defined the preprocessor, x_train_transformed, x_test_transformed, cat_columns, and num_columns

# Get the feature names after encoding
encoded_columns = list(preprocessor.named_transformers_['cat'].get_feature_names_out(cat_columns))

# Combine numeric columns with encoded columns to get all feature names
all_feat = num_columns + encoded_columns

# Convert transformed data to DataFrames with the specified column names
X_train_transformed = pd.DataFrame(x_train_transformed)
X_test_transformed = pd.DataFrame(x_test_transformed)

In [ ]:
knn = KNeighborsClassifier(n_neighbors = 10)
knn.fit(X_train_transformed, y_train)
y_pred = knn.predict(X_test_transformed)
print(f1_score(y_pred, y_test))
print(accuracy_score(y_pred, y_test))

In [ ]:
accuracies = []

# Loop through different values of n_neighbors (1 to 19)
for a in range(1, 20):
    # Create the KNN classifier with the current value of n_neighbors
    knn = KNeighborsClassifier(n_neighbors=a)
    
    # Train the KNN classifier using the training data
    knn.fit(X_train_transformed, y_train)
    
    # Make predictions on the test data
    y_pred = knn.predict(X_test_transformed)
    
    # Calculate and store the accuracy score for the current value of n_neighbors
    accuracy = accuracy_score(y_pred, y_test)
    accuracies.append(accuracy)
    
    # Optionally, you can print the accuracy for each value of n_neighbors
    print(f"Accuracy for n_neighbors = {a}: {accuracy}")

# Optionally, you can print the list of accuracy scores for all values of n_neighbors
print(accuracies)